In [35]:
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

# Project root directory
# Корневая директория проекта
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
print(DATA, DATA.exists())

/home/mchunikhin/project_steam_game_recomm_system/data True


## Reviews → Interactions

The raw review dataset is highly sparse, so we first build a clean interaction table for recommendation models.

### Processing Steps

- Keep reviews within the selected time window.
- Remove duplicate user-game pairs by keeping the most recent review.
- Filter low-activity users and low-popularity games.
- Apply iterative k-core filtering until the dataset stabilizes.

### Output

`interactions.parquet`

One row corresponds to one user-game interaction.

## Отзывы → Взаимодействия

Исходный датасет отзывов сильно разрежен, поэтому сначала формируем чистую таблицу взаимодействий для рекомендательных моделей.

### Этапы обработки

- Оставляем отзывы только за выбранный период.
- Для каждой пары пользователь–игра сохраняем только последний отзыв.
- Удаляем малоактивных пользователей и игры с небольшим числом отзывов.
- Выполняем итеративную k-core фильтрацию до стабилизации выборки.

### Результат

`interactions.parquet`

Одна строка соответствует одному взаимодействию пользователь–игра.

In [36]:
# Set True to rebuild from raw CSV, ignoring the cache
# True — пересобрать из сырых CSV, игнорируя кэш
REBUILD = False

# Filtering parameters (fixed for the project)
# Параметры фильтрации (зафиксированы для проекта)
WINDOW_START = int(datetime(2020, 1, 1, tzinfo=timezone.utc).timestamp())  # 2020-01-01
MIN_GAME_REVIEWS = 20
MIN_USER_INTER = 5
REVIEW_COLS = ["steamid", "appid", "voted_up", "playtime_forever",
               "unix_timestamp_created"]

out = DATA / "processed" / "interactions.parquet"
if out.exists() and not REBUILD:
    # Load cached interactions if available
    # Загружаем готовые взаимодействия из кэша
    inter = pd.read_parquet(out)
    print("loaded from cache:", out, f"({len(inter):,} rows)")
else:
    # Read reviews in chunks, keep only the time window and needed columns
    # Читаем отзывы чанками, оставляем только нужное окно и колонки
    parts = []
    for f in sorted((DATA / "reviews").glob("reviews-*.csv")):
        for ch in pd.read_csv(f, usecols=REVIEW_COLS, chunksize=3_000_000):
            ch = ch[ch["unix_timestamp_created"] >= WINDOW_START]
            if len(ch):
                parts.append(ch)
    inter = pd.concat(parts, ignore_index=True)
    del parts
    print("within window:", f"{len(inter):,}")

    # Remove duplicate user-game pairs, keep the most recent review
    # Удаляем дубликаты пользователь–игра, оставляем последний отзыв
    inter = (inter.sort_values("unix_timestamp_created")
                  .drop_duplicates(["steamid", "appid"], keep="last"))
    print("after dedup:", f"{len(inter):,}")

    # Iterative k-core filtering until the sets stabilize
    # Итеративная k-core фильтрация до стабилизации выборки
    while True:
        n0 = len(inter)
        g_ok = inter["appid"].value_counts()
        inter = inter[inter["appid"].isin(g_ok[g_ok >= MIN_GAME_REVIEWS].index)]
        u_ok = inter["steamid"].value_counts()
        inter = inter[inter["steamid"].isin(u_ok[u_ok >= MIN_USER_INTER].index)]
        if len(inter) == n0:
            break

    # Build the final interactions table (label = voted_up: recommended or not)
    # Формируем итоговую таблицу взаимодействий (label = voted_up: рекомендует или нет)
    inter = inter.rename(columns={"steamid": "user_id", "appid": "game_id",
                                  "voted_up": "label"})
    inter["label"] = inter["label"].astype("int8")

    # Save processed dataset
    # Сохраняем обработанный датасет
    inter.to_parquet(out, index=False)
    print("saved:", out, f"({out.stat().st_size/1e6:.1f} MB)")

print("\nSummary")
print("interactions:", f"{len(inter):,}")
print("games:       ", f"{inter['game_id'].nunique():,}")
print("users:       ", f"{inter['user_id'].nunique():,}")

loaded from cache: /home/mchunikhin/project_steam_game_recomm_system/data/processed/interactions.parquet (901,137 rows)

Summary
interactions: 901,137
games:        2,872
users:        113,552


## Games Metadata

Game metadata is used as item features for content-based models, the item tower and the ranking stage.

### Processing Steps

- Load only the required columns.
- Remove duplicate AppIDs.
- Keep only games present in the interaction dataset.
- Save the processed table.

### Note

`index_col=False` is required when reading `games.csv`, otherwise columns become misaligned.

### Output

`games.parquet`

Processed item features table.

## Метаданные игр

Метаданные игр используются как признаки для content-based моделей, item tower и этапа ранжирования.

### Этапы обработки

- Загружаем только необходимые столбцы.
- Удаляем дубликаты AppID.
- Оставляем только игры, присутствующие в таблице взаимодействий.
- Сохраняем обработанную таблицу.

### Важно

При чтении `games.csv` необходимо использовать `index_col=False`, иначе столбцы будут считаны некорректно.

### Результат

`games.parquet`

Таблица признаков игр для последующих этапов пайплайна.

In [37]:
# Columns to keep as item features
# Колонки, которые оставляем как признаки игр
GAME_COLS = ["AppID", "Name", "Release date", "Estimated owners", "Price",
             "Positive", "Negative", "Genres", "Tags", "Categories",
             "Average playtime forever", "Median playtime forever"]

out_g = DATA / "processed" / "games.parquet"
if out_g.exists() and not REBUILD:
    # Load cached games metadata if available
    # Загружаем готовые метаданные игр из кэша
    games_f = pd.read_parquet(out_g)
    print("loaded from cache:", out_g, f"({len(games_f):,} games)")
else:
    # Load games.csv (index_col=False, otherwise columns shift)
    # Читаем games.csv (index_col=False, иначе столбцы съезжают)
    games = pd.read_csv(DATA / "games" / "games.csv", usecols=GAME_COLS, index_col=False)

    # Keep only games present in interactions, drop duplicate AppIDs
    # Оставляем только игры из interactions, удаляем дубликаты AppID
    keep_ids = inter["game_id"].unique()
    games_f = (games.drop_duplicates("AppID")
               .loc[lambda d: d["AppID"].isin(keep_ids)]
               .rename(columns={"AppID": "game_id"})
               .reset_index(drop=True))

    # Save processed dataset
    # Сохраняем обработанный датасет
    games_f.to_parquet(out_g, index=False)
    print("games in interactions:", f"{len(keep_ids):,}")
    print("without metadata:     ", f"{len(set(keep_ids) - set(games_f['game_id'])):,}")
    print("saved:", out_g, f"({out_g.stat().st_size/1e6:.1f} MB)")

print("games with metadata:", f"{len(games_f):,}")

loaded from cache: /home/mchunikhin/project_steam_game_recomm_system/data/processed/games.parquet (2,759 games)
games with metadata: 2,759
